In [ ]:
# Standard libraries
import numpy as np
from pathlib import Path
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import tempfile
import os
import subprocess

# Specific libraries
import nibabel as nib
import seaborn as sns

# Import custom modules
import pp_utils as pp

In [ ]:
# Define directories
CALIB_DIR = Path("./caliber_phantom")
SCAN_DIR = CALIB_DIR / "scan_data"
OUTPUT_DIR = CALIB_DIR / "derivatives"
testname = CALIB_DIR.stem

if not CALIB_DIR.is_dir():
    raise NotADirectoryError(f"Expected a directory, got: {CALIB_DIR}") 
if not SCAN_DIR.is_dir():
    raise NotADirectoryError(f"{CALIB_DIR.stem} doesn't contain subdirectory named {SCAN_DIR.stem}") 

OUTPUT_DIR.mkdir(exist_ok=True)

### LOAD & UNPACK

In [ ]:
# Load in data and save basic features 
dicom_fnames = pp.scrape_files(SCAN_DIR, filetype=["MRe", "dcm"])
dicom_fnames.sort(key = lambda x: pp.dicom_scantime(x))
print(f"The full dataset contains {len(dicom_fnames)} DICOM files")

dataset_unpacked = list(map(pp.process_dicom, dicom_fnames))
dataset_df = pd.DataFrame(dataset_unpacked)
print("Scans unpacked into pandas DataFrame structure")

In [ ]:
def plot_num_of_repeats(df, col_name):

    num_repeats = df.groupby(col_name).size().rename("count").reset_index()

    fig, ax = plt.subplots(figsize=(16, 9))
    sns.barplot(data=num_repeats, x=col_name, y="count", hue=col_name, palette="husl", ax=ax)

    # Tidy up axes
    ax.set_xticks(np.arange(num_repeats.values.shape[0]), num_repeats[col_name])
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha="center")
    ax.set_yticks(range(0, num_repeats["count"].values.max() + 1))
    ax.set_ylabel("Number of Repeats")
    fig.tight_layout()

    plt.show()

plot_num_of_repeats(dataset_df, "seq_name")

### REGISTRATION

In [ ]:
# Extract images from dataframe
image_4d = np.stack(dataset_df["image"].values, axis=-1)
aff_0 = dataset_df["affine"].iloc[0]
spacing_0 = dataset_df["spacing"].iloc[0]

# Extract b-values in order to be able to identify b=0 scan to register other scans to

bvals = dataset_df["b_value"].values

# Perform registration:
# function returns a registered 4D image array and also
# a numpy array of indices for which registration failed
reg_4d, i_drop = pp.register_to_b0(data_4d=image_4d, 
                                   bvals=bvals, 
                                   spacing=spacing_0,
                                   b0_threshold=5)

# Create a new dataframe with only the measurements that were successfully
# registered and append a column for registered scans
reg_df = dataset_df.copy()
reg_4d_reshuffled = np.transpose(reg_4d, (3,0,1,2))
reg_df["reg_image"] = list(reg_4d_reshuffled)

### CREATE b=0 MASK

In [ ]:
brainmask = None
brain = "empty"

while brainmask is None:
    if brain.upper() == "Y":
        brainmask = True
    elif brain.upper() == "N":
        brainmask = False
    else:
        brain = input("Is target image a brain (Y/N)?")

print(f"Subject is a brain: {str(brainmask)}")

In [ ]:
def run_bet_on_array(vol: np.ndarray, affine: np.ndarray | None = None, frac: float = 0.5):

    with tempfile.TemporaryDirectory() as tmpdir:
        in_path = os.path.join(tmpdir, "input.nii.gz")
        out_base = os.path.join(tmpdir, "brain")

        img = nib.Nifti1Image(vol.astype(np.float32), affine)
        nib.save(img, in_path)

        cmd = ["bet", in_path, out_base, "-f", str(frac)]

        subprocess.run(cmd, capture_output=True, text=True)
        brain = nib.load(out_base + ".nii.gz").get_fdata()

        return brain

b0_threshold = 5
b0_scans = np.stack(reg_df[reg_df["b_value"] < b0_threshold]["image"].values, axis=-1)
b0_avg = np.mean(b0_scans, axis=-1)

if brainmask:
    # If target data is a brain we can run BET2 to extract the brain region
    mask = run_bet_on_array(b0_avg, affine=aff_0, frac=0.73).astype(bool)

else:
    # If a phantom we shall just use ants to create the mask 
    brain = pp.get_mask(b0_avg)
    mask = brain > 0

# Plot the resulting masked data
plotdata = b0_avg
plotdata[mask==0] = 0
zmin = np.percentile(plotdata[mask], 1)
zmax = np.percentile(plotdata[mask], 99)
px.imshow(plotdata, animation_frame=2, zmin=zmin, zmax=zmax, height=800, width=1000, title="Masked b=0 image")

## SAVE OUTPUTS

In [ ]:
output_df = reg_df.groupby(by="seq_name").first().reset_index()
print(output_df.shape)

raw_images = np.stack(output_df["image"].values, axis=-1)
raw_nii = nib.Nifti1Image(raw_images, affine=aff_0)
pp.save_nifti(raw_nii, OUTPUT_DIR, testname + "_" + "RAW.nii.gz")

reg_images = np.stack(output_df["reg_image"].values, axis=-1)
reg_nii = nib.Nifti1Image(reg_images, affine=aff_0)
pp.save_nifti(reg_nii, OUTPUT_DIR, testname + "_" + "RAW_REG.nii.gz")

b0_mask_nii = nib.Nifti1Image(mask.astype(float), aff_0)
pp.save_nifti(b0_mask_nii, OUTPUT_DIR, testname + "_" + "MASK.nii.gz")

param_df = output_df.drop(["image", "reg_image"], axis=1, inplace=False)
param_fname = OUTPUT_DIR / (testname + "_" + "params.h5")
param_df.to_hdf(param_fname, key="df", mode="w", index=False)